In [ ]:
import torch
import numpy as np
import tempfile
import re
from pathlib import Path
from datasets import load_dataset
from IPython.display import Video, display
from safetensors.torch import load_file

from src.config import Config
from src.model.virality_predictor import ViralityPredictor
from src.model.data_processor import DataProcessor

device = torch.device('mps' if torch.backends.mps.is_available(
) else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

config = Config()
checkpoint_dir = Path(config.checkpoint_path)
checkpoint_folders = [f for f in checkpoint_dir.iterdir(
) if f.is_dir() and re.match(r'checkpoint-\d+$', f.name)]
latest_checkpoint = max(
    checkpoint_folders, key=lambda f: int(f.name.split('-')[1]))
print(f'Loading model from: {latest_checkpoint}')

model = ViralityPredictor(config)
state_dict = load_file(latest_checkpoint / 'model.safetensors')
model.load_state_dict(state_dict)
model.to(device)
model.eval()

assert model.tabular_means is not None and model.tabular_stds is not None

processor = DataProcessor(config)
processor.tabular_means = torch.tensor(model.tabular_means.cpu())
processor.tabular_stds = torch.tensor(model.tabular_stds.cpu())

dataset = load_dataset(config.dataset_id)['train']
example_idx = np.random.randint(0, len(dataset))
example = dataset[example_idx]

print(f'\n=== Ground Truth (Example {example_idx}) ===')
print(f'Engagement Score: {example["engagement_score"]:.2f}')
print(f'Velocity Score:   {example["view_velocity_score"]:.2f}')
print(f'Is Viral:         {bool(example["is_viral"])}')

batch = {k: [v] for k, v in example.items()}
processed = processor._process_batch(batch)
inference_batch = {k: v.to(device)
                   for k, v in processed.items() if k != 'labels'}

predictions = model.predict_scores(**inference_batch)

prob = predictions["viral_prob"].item()
PRECISION_THRESHOLD = 0.7

print(f'\n=== Model Predictions (Inverse Log Space) ===')
print(f'Pred Engagement: {predictions["engagement"].item():.2f}')
print(f'Pred Velocity:   {predictions["velocity"].item():.2f}')
print(f'Viral Confidence: {prob:.2%}')
print(
    f'Classification:  {"VIRAL" if prob >= PRECISION_THRESHOLD else "NOT VIRAL"}')

if example['video_bytes']:
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
        tmp.write(example['video_bytes'])
        display(Video(tmp.name, embed=True, width=300))

Using device: mps
Loading model from: data/checkpoints/checkpoint-5


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.norm.weight                                                  | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias   


=== Ground Truth (Example 9030) ===
Engagement Score: 0.17
Velocity Score:   23.34
Is Viral:         False
{'input_ids': tensor([[  101,   101,  1031,  2516,  1033,  2007,  2008,  9273,  3388,  2053,
          2028, 21645,  1001, 25210, 15007,  1001, 21348, 11823, 15007,  1001,
         25210,  8445,  2923,  1001, 25210, 18715,  1001, 25210, 10288, 29048,
          2015,  1001, 25210, 15007,  1001, 25210,  8445,  2923,  1031,  2189,
          1033,  2434,  2614,  2011,  4581,  9594,  1031,  7860,  1033,  1031,
          1000, 25210, 15007,  1000,  1010,  1000, 21348, 11823, 15007,  1000,
          1010,  1000, 25210,  8445,  2923,  1000,  1010,  1000, 25210, 18715,
          1000,  1010,  1000, 25210, 10288, 29048,  2015,  1000,  1033,  1031,
          5200,  1033,  2711,  1031,  8840,  2278,  1033,  3050,  3349,  3050,
          3349,  2662,  3379,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         